In [1]:
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.layers import LSTM, Dense, Activation, Dropout, BatchNormalization

2024-12-15 20:30:39.414501: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-12-15 20:30:39.424873: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1734282039.436601 1536809 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1734282039.440158 1536809 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-12-15 20:30:39.452775: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
filepath = 'poems.txt'

text = open(filepath, 'rb').read().decode(encoding='utf-8').lower()[:10_000]
characters = sorted(set(text))

char_to_index = dict((c, i) for i, c in enumerate(characters))
index_to_char = dict((i, c) for i, c in enumerate(characters))

SEQ_LENGTH = 40
STEP_SIZE = 3

In [3]:
sentences = []
next_char = []
for i in range(0, len(text) - SEQ_LENGTH, STEP_SIZE):
    sentences.append(text[i: i + SEQ_LENGTH])
    next_char.append(text[i + SEQ_LENGTH])

x = np.zeros((len(sentences), SEQ_LENGTH, len(characters)), dtype=np.bool)
y = np.zeros((len(sentences), len(characters)), dtype=np.bool)

for i, satz in enumerate(sentences):
    for t, char in enumerate(satz):
        x[i, t, char_to_index[char]] = 1
    y[i, char_to_index[next_char[i]]] = 1

In [4]:
model = Sequential()
model.add(LSTM(128, return_sequences=True, input_shape=(SEQ_LENGTH, len(characters))))
model.add(Dropout(0.2))
model.add(BatchNormalization())
model.add(LSTM(128))
model.add(Dropout(0.2))
model.add(BatchNormalization())
model.add(Dense(128))
model.add(Dense(len(characters)))
model.add(Activation('softmax'))

I0000 00:00:1734282042.904978 1536809 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4309 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 6GB Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6
/media/aghabidareh/b8b65723-1e2e-4a2c-a0f2-1b84824ccd411/projects/a/venv/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [5]:
model.compile(loss='categorical_crossentropy', optimizer=RMSprop(learning_rate=0.001))

In [7]:
model.fit(x, y, batch_size=64, epochs=100)

Epoch 1/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 1.6787 
Epoch 2/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 1.5237 
Epoch 3/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1.4420 
Epoch 4/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1.3265 
Epoch 5/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1.2334 
Epoch 6/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 1.1365 
Epoch 7/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1.0338 
Epoch 8/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.9644 
Epoch 9/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.9158 
Epoch 10/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.8324 
Epoch 11/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.7400 
Epoch 12/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.7120 
Epoch 13/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.6575 
Epoch 14/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.6316 
Epoch 15/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - los

In [8]:
def sample(preds, temperature=1.0):
    preds = np.asarray(preds).astype('float64')
    preds = np.log(preds) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    probas = np.random.multinomial(1, preds, 1)
    return np.argmax(probas)

In [9]:
def generate_text(length, temperature=1.0):
    start_index = random.randint(0, len(text) - SEQ_LENGTH - 1)
    generated = ''
    sentence = text[start_index: start_index + SEQ_LENGTH]
    generated += sentence
    for i in range(length):
        x_predictions = np.zeros((1, SEQ_LENGTH, len(characters)))
        for t, char in enumerate(sentence):
            x_predictions[0, t, char_to_index[char]] = 1

        predictions = model.predict(x_predictions, verbose=0)[0]
        next_index = sample(predictions, temperature)
        next_character = index_to_char[next_index]

        generated += next_character
        sentence = sentence[1:] + next_character
    return generated

In [10]:
generated_text = generate_text(100, temperature=0.7)
print(generated_text)

ین گل ز بر همنفسی می‌آید
شادی به دلم از جای خوان ین بیست عآ
پنو کو که ره حزف بابتد
خشو جد در جانای به به نخش وین برو باناونسه
که که سر طحف د


In [11]:
model.save('model.h5')